# SPATIAL INTELLIGENCE — DUBAI TOWER (Multi-Floor Connected Building)

Spatial-Intelligence analysis of **two stacked residential floors** of the Dubai Tower —
**Level 03** (`assets/obj/F03.obj`, lower) and **Level 04** (`assets/obj/F04.obj`, upper) —
treated as **one connected building graph**, replicating the *Cité Radieuse / Unité d'Habitation*
workflow.

**Key idea (same as Cité Radieuse).** Every metric is computed on a **single connected
building graph** (both floors joined through vertical stair/core edges), not per floor in
isolation. The two floors come from **two separate OBJ files** (each a flat plan) and are
stacked artificially in Z for the graph and the 3D views.

**Method (Therme-Vals / grid-sampling approach).**
1. Import each floor OBJ → collect triangulated faces.
2. Lay a regular grid (`GRID_SIZE`) over the shared bounding box; keep only points inside the
   mesh (*navigable area*) via a vectorised point-in-triangle test.
3. Each valid point → a topologic vertex at `Z = floor_index × FLOOR_HEIGHT`, plus a flat
   display cell for the 2D heatmaps.
4. Connect 4-neighbours within each floor → horizontal edges.
5. Snap each stair/core location to the closest navigable node on each floor → vertical edges.
6. `Graph.ByVerticesEdges` → the combined **building graph**; run all metrics on it.

> **Stairs not placed yet.** The apartment cores will be added later (currently only in a 3DM).
> Until then, the notebook **auto-places** a few vertical connectors on the shared navigable
> footprint so the two floors connect and the whole pipeline runs end-to-end. Replace
> `STAIR_LOCATIONS` with the real core coordinates once the apartments OBJ is available.

## 1. Import the needed libraries

In [1]:
import os, math, time
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Render every figure as a STATIC image via kaleido (no WebGL) — avoids VS Code's
# "WebGL is not supported" crash for the heatmap cells.
pio.renderers.default = "png"

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy version

In [2]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This notebook requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.43) is OLDER than the latest version (0.9.50) from PyPI. Please consider upgrading to the latest version.


## 3. Configuration

In [3]:
from pathlib import Path

renderer = "png"               # "png" renders via kaleido (no WebGL)
INTERACTIVE_RENDERER = "png"   # set to "browser" for rotatable 3D figures

# --- Floors (bottom -> top) --------------------------------------------------
# Each floor is a SEPARATE OBJ file (unlike Cité Radieuse where one OBJ held all
# floors). FLOOR_LEVELS are just stack-keys; the real Z is assigned by index.
FLOOR_TAGS   = ["F03", "F04"]                       # bottom -> top
FLOOR_NAMES  = ["Level 03 (lower)", "Level 04 (upper)"]
FLOOR_LEVELS = list(range(len(FLOOR_TAGS)))         # [0, 1] stack keys

# In-plane axes AFTER Topology.ByOBJPath. The Dubai OBJs were rotated so the plan
# imports flat in X-Y (Z ~ constant) — the standard orientation.
PLAN_AXES = ("X", "Y")

# Resolve <repo>/assets/obj/<tag>.obj robustly (CWD = this notebook's folder).
def _find_obj(tag):
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = base / "assets" / "obj" / f"{tag}.obj"
        if p.exists():
            return p
    hits = list(Path.cwd().parent.glob(f"**/assets/obj/{tag}.obj"))
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Could not find assets/obj/{tag}.obj under the project root")

FLOOR_OBJS = [str(_find_obj(t)) for t in FLOOR_TAGS]
for t, p in zip(FLOOR_TAGS, FLOOR_OBJS):
    print(f"  {t}: {p}")

# Analysis grid spacing (model units ~ metres). 4.0 = quick smoke test (~160 nodes);
# 1.5 = good resolution (~1200 nodes). Drop lower for finer maps (slower betweenness).
GRID_SIZE = 1.5

# Vertical spacing used to STACK the floors in the combined 3D graph (visual only).
FLOOR_HEIGHT = 10.0

# --- Stair / vertical-core locations in PLAN coordinates (X, Y) ---------------
# Floor indices: 0 = Level 03 (bottom), 1 = Level 04 (top).
#   (x, y)                       -> connects every adjacent floor pair
#   (x, y, [(floor_a, floor_b)]) -> connects only the listed pairs
# Leave EMPTY to auto-place connectors on the shared navigable footprint so the
# pipeline runs before the real apartment cores are added.
STAIR_LOCATIONS = []           # e.g. [(80.0, 0.0), (60.0, -8.0, [(0, 1)])]
N_AUTO_STAIRS   = 4            # how many connectors to auto-place when the list is empty

# Output folder for PNGs + summary, named with the grid size of this run.
BASE_DIR   = str(Path.cwd())
ASSETS_DIR = os.path.join(BASE_DIR, "outputs", f"dubai_multifloor_{GRID_SIZE}m-grid")
os.makedirs(ASSETS_DIR, exist_ok=True)
print("ASSETS_DIR:", ASSETS_DIR)

SAVE_IMAGES = True

def save_fig(fig, filename):
    if not SAVE_IMAGES or fig is None:
        return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        w = int(fig.layout.width)  if fig.layout.width  else 1800
        h = int(fig.layout.height) if fig.layout.height else 1100
        fig.write_image(path, width=w, height=h, scale=2)
        print(f"Saved: {path}  ({w}x{h})")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

  F03: c:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\F03.obj
  F04: c:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\F04.obj
ASSETS_DIR: c:\Users\Win11\GraphML_RaniaChihaoui\DubaiTower\outputs\dubai_multifloor_1.5m-grid


## 4. Utility functions

* `extract_triangles` / `points_inside` — geometry helpers (pull plan triangles via `PLAN_AXES`, point-in-mesh test).
* `find_closest_node` — snaps a stair location to the nearest grid node (the instructor's `find_closest_vertex`).
* `show_grid_heatmaps` — the metric heatmaps as 2D `go.Heatmap` panels (one per floor) on a black background with real plan coordinates.

In [4]:
_AX = {"X": Vertex.X, "Y": Vertex.Y, "Z": Vertex.Z}
_GU, _GV = _AX[PLAN_AXES[0]], _AX[PLAN_AXES[1]]

def extract_triangles(face_list):
    # Return (T,3,2) array of plan-space triangles (using PLAN_AXES) for a list of faces.
    tris = []
    for f in face_list:
        vs = Topology.Vertices(f)
        pts = [(_GU(v), _GV(v)) for v in vs]
        for i in range(1, len(pts) - 1):       # fan-triangulate (faces are already triangles)
            tris.append([pts[0], pts[i], pts[i + 1]])
    return np.array(tris)

def points_inside(tris, P):
    # Boolean mask: which points in P (N,2) fall inside ANY triangle of tris (T,3,2).
    a, b, c = tris[:, 0], tris[:, 1], tris[:, 2]
    v0 = b - a; v1 = c - a
    d00 = (v0 * v0).sum(1); d01 = (v0 * v1).sum(1); d11 = (v1 * v1).sum(1)
    den = d00 * d11 - d01 * d01
    den[den == 0] = 1e-12
    inside = np.zeros(len(P), bool)
    for i, p in enumerate(P):
        v2 = p - a
        d20 = (v2 * v0).sum(1); d21 = (v2 * v1).sum(1)
        u = (d11 * d20 - d01 * d21) / den
        w = (d00 * d21 - d01 * d20) / den
        if np.any((u >= -1e-6) & (w >= -1e-6) & (u + w <= 1 + 1e-6)):
            inside[i] = True
    return inside

def find_closest_node(node_xy, x, y):
    d = (node_xy[:, 0] - x) ** 2 + (node_xy[:, 1] - y) ** 2
    return int(d.argmin())

def rk(u, v):
    return (round(float(u), 3), round(float(v), 3))

def make_cell_face(cx, cy, h):
    pts = [Vertex.ByCoordinates(cx - h, cy - h, 0.0), Vertex.ByCoordinates(cx + h, cy - h, 0.0),
           Vertex.ByCoordinates(cx + h, cy + h, 0.0), Vertex.ByCoordinates(cx - h, cy + h, 0.0)]
    return Face.ByWire(Wire.ByVertices(pts, close=True))

# ---------------------------------------------------------------------------
# show_grid_heatmaps — metric maps as 2D go.Heatmap panels (one per floor) laid
# out HORIZONTALLY on a black background, each with REAL plan coordinates.
#   per_floor_xyv : list (len = n floors) of (xs, ys, vals)
# ---------------------------------------------------------------------------
def show_grid_heatmaps(per_floor_xyv, title, filename, colorScale="viridis", grid_size=None):
    if grid_size is None:
        grid_size = GRID_SIZE
    cmap = {"viridis": "Viridis", "thermal": "Inferno",
            "plasma": "Plasma", "rainbow": "Rainbow"}.get(colorScale, "Viridis")

    nonempty = [np.asarray(v, float) for (_, _, v) in per_floor_xyv if len(v)]
    allv = np.concatenate(nonempty) if nonempty else np.array([0.0, 1.0])
    mn, mx = float(np.min(allv)), float(np.max(allv))
    if mx == mn:
        mx = mn + 1e-9

    ux = np.arange(UMIN, UMAX + grid_size, grid_size)
    uy = np.arange(VMIN, VMAX + grid_size, grid_size)

    fig = make_subplots(rows=1, cols=len(FLOOR_LEVELS), horizontal_spacing=0.06,
                        subplot_titles=[FLOOR_NAMES[i] for i in range(len(FLOOR_LEVELS))])
    for fi, (xs, ys, vals) in enumerate(per_floor_xyv):
        Z = np.full((len(uy), len(ux)), np.nan)
        if len(vals):
            xs = np.asarray(xs, float); ys = np.asarray(ys, float); vals = np.asarray(vals, float)
            xi = np.round((xs - UMIN) / grid_size).astype(int)
            yi = np.round((ys - VMIN) / grid_size).astype(int)
            ok = (xi >= 0) & (xi < len(ux)) & (yi >= 0) & (yi < len(uy))
            Z[yi[ok], xi[ok]] = vals[ok]
        fig.add_trace(go.Heatmap(
            x=ux, y=uy, z=Z, colorscale=cmap, zmin=mn, zmax=mx,
            showscale=(fi == len(FLOOR_LEVELS) - 1), hoverongaps=False,
            colorbar=dict(tickfont=dict(color="white"),
                          outlinecolor="rgba(255,255,255,0.3)")),
            row=1, col=fi + 1)
        fig.update_yaxes(scaleanchor=f"x{fi+1 if fi else ''}", scaleratio=1, row=1, col=fi + 1)

    dw = UMAX - UMIN; dh = VMAX - VMIN
    px_pu = max(8, min(40, int(1500 / (dw * len(FLOOR_LEVELS) + 1))))
    fig_w = max(700, int(dw * px_pu) * len(FLOOR_LEVELS) + 300)
    fig_h = max(420, int(dh * px_pu) + 170)

    fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
    for ann in fig.layout.annotations:
        ann.font.color = "white"
    fig.update_layout(title=dict(text=title, font=dict(color="white")),
                      height=fig_h, width=fig_w, paper_bgcolor="black", plot_bgcolor="black")
    fig.show(renderer=renderer)
    save_fig(fig, filename)
    return fig

## 5. Import the two floor OBJs and compute the shared plan box

Each floor is loaded from its own OBJ. `Topology.ByOBJPath` returns clusters of triangulated
faces; we collect every face per floor and cache its plan triangles. The two floors are then
overlaid in one shared bounding box (they share the Dubai Tower footprint).

In [ ]:
def load_floor_faces(path):
    result = Topology.ByOBJPath(path)
    items = result if isinstance(result, list) else [result]
    fs = []
    for item in items:
        if Topology.IsInstance(item, "Cluster"):
            cf = Cluster.Faces(item)
            if cf: fs.extend(cf)
        elif Topology.IsInstance(item, "Face"):
            fs.append(item)
    return fs

floor_faces = {}     # level -> list of faces
floor_tris  = {}     # level -> (T,3,2) plan triangles (cached)
for fi, lv in enumerate(FLOOR_LEVELS):
    t0 = time.time()
    fs = load_floor_faces(FLOOR_OBJS[fi])
    floor_faces[lv] = fs
    floor_tris[lv]  = extract_triangles(fs)
    print(f"  {FLOOR_NAMES[fi]} [{FLOOR_TAGS[fi]}]: {len(fs)} faces  ({time.time()-t0:.1f}s)")

# Shared plan bounding box across both floors
allxy = np.vstack([floor_tris[lv].reshape(-1, 2) for lv in FLOOR_LEVELS])
UMIN, VMIN = allxy.min(0)
UMAX, VMAX = allxy.max(0)
UMID, VMID = 0.5 * (UMIN + UMAX), 0.5 * (VMIN + VMAX)
COLGAP = (UMAX - UMIN) + 8.0   # horizontal gap between floor panels (display only)
print(f"Shared plan box: u[{UMIN:.1f},{UMAX:.1f}]  v[{VMIN:.1f},{VMAX:.1f}]  "
      f"({UMAX-UMIN:.1f} x {VMAX-VMIN:.1f})")

  Level 03 (lower) [F03]: 12822 faces  (32.3s)


## 6. Show the two raw floor plans

In [ ]:
fig = make_subplots(rows=1, cols=len(FLOOR_LEVELS), horizontal_spacing=0.05,
                    subplot_titles=[FLOOR_NAMES[i] for i in range(len(FLOOR_LEVELS))])
for i, lv in enumerate(FLOOR_LEVELS):
    for t in floor_tris[lv]:
        xs = list(t[:, 0]) + [t[0, 0]]
        ys = list(t[:, 1]) + [t[0, 1]]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", fill="toself",
                                 line=dict(color="rgba(170,195,255,0.55)", width=0.4),
                                 fillcolor="rgba(70,120,235,0.35)", showlegend=False), row=1, col=i + 1)
    fig.update_yaxes(scaleanchor=f"x{i+1 if i else ''}", scaleratio=1, row=1, col=i + 1)

dw = UMAX - UMIN; dh = VMAX - VMIN
px_pu = max(8, min(40, int(1500 / (dw * len(FLOOR_LEVELS) + 1))))
fig_w = max(700, int(dw * px_pu) * len(FLOOR_LEVELS) + 240)
fig_h = max(420, int(dh * px_pu) + 170)

fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
for ann in fig.layout.annotations:
    ann.font.color = "white"
fig.update_layout(title=dict(text="Two imported floor plans (plan view)", font=dict(color="white")),
                  height=fig_h, width=fig_w, paper_bgcolor="black", plot_bgcolor="black")
fig.show(renderer=renderer)
save_fig(fig, "01_floor_plans.png")

## 7. Sample a navigable grid on each floor

A regular grid is laid over the shared bounding box; only points inside the meshed (navigable)
area are kept. These become the graph nodes of each floor.

In [ ]:
us = np.arange(UMIN, UMAX + GRID_SIZE, GRID_SIZE)
vs = np.arange(VMIN, VMAX + GRID_SIZE, GRID_SIZE)
UU, VV = np.meshgrid(us, vs)
GRID_PTS = np.column_stack([UU.ravel(), VV.ravel()])

floor_valid = {}     # level -> valid_xy ndarray
for i, lv in enumerate(FLOOR_LEVELS):
    floor_valid[lv] = GRID_PTS[points_inside(floor_tris[lv], GRID_PTS)]
    print(f"  {FLOOR_NAMES[i]}: {len(floor_valid[lv])} navigable nodes")

## 8. Resolve stair / vertical-core locations

Uses the manual `STAIR_LOCATIONS` if provided; otherwise **auto-places** `N_AUTO_STAIRS`
connectors on the shared navigable footprint (positions navigable on *both* floors), so the
two levels connect and the pipeline runs before the real apartment cores are added.

In [ ]:
def _auto_stairs(n):
    # Pick points that are navigable on EVERY floor, spread across the footprint.
    common_keys = None
    for lv in FLOOR_LEVELS:
        ks = set(rk(u, v) for (u, v) in floor_valid[lv])
        common_keys = ks if common_keys is None else (common_keys & ks)
    common = sorted(common_keys)
    if not common:
        print("  WARNING: floors share no navigable grid cells — cannot auto-place stairs.")
        return []
    step = max(1, len(common) // n)
    return [(float(u), float(v)) for (u, v) in common[::step][:n]]

if STAIR_LOCATIONS:
    stairs_used = STAIR_LOCATIONS
    print(f"Using {len(stairs_used)} manual stair location(s).")
else:
    auto = _auto_stairs(N_AUTO_STAIRS)
    stairs_used = [(x, y, [(0, 1)]) for (x, y) in auto]   # connect Level 03 <-> Level 04
    print(f"AUTO-placed {len(stairs_used)} stair connector(s) on the shared footprint "
          f"(replace STAIR_LOCATIONS with real cores later):")
    for s in stairs_used:
        print(f"    ({s[0]:.2f}, {s[1]:.2f})")

## 9. Show the navigable grids + stair locations

In [ ]:
fig = make_subplots(rows=1, cols=len(FLOOR_LEVELS), horizontal_spacing=0.05,
                    subplot_titles=[FLOOR_NAMES[i] for i in range(len(FLOOR_LEVELS))])
sx = [s[0] for s in stairs_used]; sy = [s[1] for s in stairs_used]
for i, lv in enumerate(FLOOR_LEVELS):
    valid = floor_valid[lv]
    fig.add_trace(go.Scatter(x=valid[:, 0], y=valid[:, 1], mode="markers",
                             marker=dict(size=5, color="royalblue"), showlegend=False), row=1, col=i + 1)
    fig.add_trace(go.Scatter(x=sx, y=sy, mode="markers",
                             marker=dict(size=13, color="red", symbol="x"), name="stairs",
                             showlegend=(i == 0)), row=1, col=i + 1)
    fig.update_yaxes(scaleanchor=f"x{i+1 if i else ''}", scaleratio=1, row=1, col=i + 1)

dw = UMAX - UMIN; dh = VMAX - VMIN
px_pu = max(8, min(40, int(1500 / (dw * len(FLOOR_LEVELS) + 1))))
fig_w = max(700, int(dw * px_pu) * len(FLOOR_LEVELS) + 240)
fig_h = max(420, int(dh * px_pu) + 170)

fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
for ann in fig.layout.annotations:
    ann.font.color = "white"
fig.update_layout(title=dict(text="Navigable grids + stair locations (red x)", font=dict(color="white")),
                  height=fig_h, width=fig_w, paper_bgcolor="black", plot_bgcolor="black",
                  legend=dict(font=dict(color="white")))
fig.show(renderer=renderer)
save_fig(fig, "02_navigable_grids.png")

## 10. Build the per-floor graphs, the display cells, and stack everything

For each floor the valid points become topologic vertices at `Z = floor_index * FLOOR_HEIGHT`
(for the 3D graph) and a flat display cell (for the 2D heatmaps). Horizontal edges join
4-neighbour valid points within each floor.

In [ ]:
all_v = []            # topologic vertices (all floors, stacked in Z)
all_e = []            # topologic edges
floor_index_map = {}  # level -> {(round u, round v): global vertex index}
H = GRID_SIZE / 2.0

for fi, lv in enumerate(FLOOR_LEVELS):
    valid = floor_valid[lv]
    z = fi * FLOOR_HEIGHT
    idx = {}
    for (u, v) in valid:
        idx[rk(u, v)] = len(all_v)
        all_v.append(Vertex.ByCoordinates(float(u), float(v), float(z)))
    floor_index_map[lv] = idx
    ne = 0
    for (u, v) in valid:
        for du, dv in [(GRID_SIZE, 0), (0, GRID_SIZE)]:
            k = rk(u + du, v + dv)
            if k in idx:
                all_e.append(Edge.ByVertices([all_v[idx[rk(u, v)]], all_v[idx[k]]]))
                ne += 1
    print(f"  {FLOOR_NAMES[fi]}: {len(valid)} nodes, {ne} horizontal edges")
print(f"Subtotal: {len(all_v)} nodes, {len(all_e)} horizontal edges")

def heatmap_from_graph(values, title, filename, colorScale):
    # Gather real plan coordinates per floor (by node Z), then render the 2D heatmaps.
    per_floor = []
    for fi in range(len(FLOOR_LEVELS)):
        xs, ys, vals = [], [], []
        for v, val in zip(gverts, values):
            if int(round(Vertex.Z(v) / FLOOR_HEIGHT)) == fi:
                xs.append(Vertex.X(v)); ys.append(Vertex.Y(v)); vals.append(val)
        per_floor.append((xs, ys, vals))
    return show_grid_heatmaps(per_floor, title, filename, colorScale, grid_size=GRID_SIZE)

## 11. Connect the floors through the stairs

For every stair location and adjacent floor pair we snap to the closest navigable node on each
floor (`find_closest_node`) and add a **vertical stair edge** — turning two separate plans into
one connected building.

In [ ]:
stair_node_pairs = []
for stair in stairs_used:
    sx_, sy_ = stair[0], stair[1]
    pairs = stair[2] if len(stair) >= 3 and stair[2] else [(i, i + 1) for i in range(len(FLOOR_LEVELS) - 1)]
    for fa, fb in pairs:
        a, b = FLOOR_LEVELS[fa], FLOOR_LEVELS[fb]
        va, vb = floor_valid[a], floor_valid[b]
        ia = find_closest_node(va, sx_, sy_)
        ib = find_closest_node(vb, sx_, sy_)
        gia = floor_index_map[a][rk(va[ia, 0], va[ia, 1])]
        gib = floor_index_map[b][rk(vb[ib, 0], vb[ib, 1])]
        all_e.append(Edge.ByVertices([all_v[gia], all_v[gib]]))
        stair_node_pairs.append((gia, gib))
print(f"Added {len(stair_node_pairs)} vertical stair edges from {len(stairs_used)} location(s)")

## 12. Build the combined BUILDING graph

In [ ]:
t0 = time.time()
building_graph = Graph.ByVerticesEdges(all_v, all_e)
gverts = Graph.Vertices(building_graph)
gedges = Graph.Edges(building_graph)
print(f"Building graph: {len(gverts)} vertices, {len(gedges)} edges  ({time.time()-t0:.1f}s)")
print(f"Graph density:  {Graph.Density(building_graph):.5f}")

## 13. Show the combined 3D building graph

Two stacked floors connected by the vertical stair edges (highlighted in red).

In [ ]:
fig = Topology.Show(building_graph,
                   vertexSize=3, vertexColor="royalblue",
                   edgeColor="lightgrey", edgeWidth=1,
                   backgroundColor="black", width=1400, height=900,
                   showFigure=False, renderer=renderer)
for (a, b) in stair_node_pairs:
    pa, pb = all_v[a], all_v[b]
    fig.add_trace(go.Scatter3d(x=[Vertex.X(pa), Vertex.X(pb)], y=[Vertex.Y(pa), Vertex.Y(pb)],
                               z=[Vertex.Z(pa), Vertex.Z(pb)], mode="lines",
                               line=dict(color="red", width=6), showlegend=False))
fig.update_layout(
    scene=dict(aspectmode="data",
               xaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="x", font=dict(color="white"))),
               yaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="y", font=dict(color="white"))),
               zaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="z (floor stack)", font=dict(color="white")))),
    scene_camera=dict(projection=dict(type="orthographic"), eye=dict(x=1.4, y=1.4, z=1.4)),
    scene_dragmode="orbit")
fig.show(renderer=INTERACTIVE_RENDERER)
save_fig(fig, "03_building_graph_3d.png")

## 14. Minimum Spanning Tree

The MST connects all building nodes with the fewest edges / minimum total weight — the
irreducible circulation skeleton. Comparing MST density to the full-graph density reveals how
much **redundancy** (alternative paths / loops) the layout provides.

In [ ]:
t0 = time.time()
mst = Graph.MinimumSpanningTree(building_graph)
mst_verts = Graph.Vertices(mst)
mst_edges = Graph.Edges(mst)
print(f"Minimum Spanning Tree: {len(mst_verts)} vertices, {len(mst_edges)} edges  ({time.time()-t0:.1f}s)")
print(f"Original graph — Density: {Graph.Density(building_graph):.5f}")
print(f"MST            — Density: {Graph.Density(mst):.5f}")

edge_x, edge_y, edge_z = [], [], []
for e in mst_edges:
    vs_e = Topology.Vertices(e)
    if len(vs_e) >= 2:
        edge_x += [Vertex.X(vs_e[0]), Vertex.X(vs_e[1]), None]
        edge_y += [Vertex.Y(vs_e[0]), Vertex.Y(vs_e[1]), None]
        edge_z += [Vertex.Z(vs_e[0]), Vertex.Z(vs_e[1]), None]
vx = [Vertex.X(v) for v in mst_verts]
vy = [Vertex.Y(v) for v in mst_verts]
vz = [Vertex.Z(v) for v in mst_verts]

_axx = dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="x", font=dict(color="white")))
_axy = dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="y", font=dict(color="white")))
_axz = dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="z (floor stack)", font=dict(color="white")))

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode="lines",
                           line=dict(color="lightgrey", width=1), showlegend=False))
fig.add_trace(go.Scatter3d(x=vx, y=vy, z=vz, mode="markers",
                           marker=dict(size=2, color="royalblue"), showlegend=False))
fig.update_layout(title=dict(text="Minimum Spanning Tree — whole building", font=dict(color="white")),
                  paper_bgcolor="black",
                  scene=dict(aspectmode="data", xaxis=_axx, yaxis=_axy, zaxis=_axz),
                  scene_camera=dict(projection=dict(type="orthographic"), eye=dict(x=1.4, y=1.4, z=1.4)),
                  scene_dragmode="orbit", width=1400, height=900)
fig.show(renderer=INTERACTIVE_RENDERER)
save_fig(fig, "10_mst_3d.png")

## 15. Cross-floor Shortest Path (+ Straightened)

Navigate from one end of the **bottom** floor to the far end of the **top** floor. The route
climbs through a stair node, demonstrating genuine cross-floor connectivity. The simplified
path keeps only floor-transition (stair) waypoints.

In [ ]:
def graph_closest(x, y, z):
    best, bd = None, 1e18
    for v in gverts:
        d = (Vertex.X(v) - x)**2 + (Vertex.Y(v) - y)**2 + (Vertex.Z(v) - z)**2
        if d < bd: bd, best = d, v
    return best

start_v = graph_closest(UMIN + GRID_SIZE, VMID, 0)
end_v   = graph_closest(UMAX - GRID_SIZE, VMID, (len(FLOOR_LEVELS) - 1) * FLOOR_HEIGHT)

t0 = time.time()
path = Graph.ShortestPath(building_graph, vertexA=start_v, vertexB=end_v)
key_verts = None
simp_len  = 0.0

if path is None:
    print("No path found — check STAIR_LOCATIONS / auto-stairs.")
else:
    path_len     = Wire.Length(path)
    path_verts_p = Topology.Vertices(path)
    path_edges_p = Topology.Edges(path)
    n_turns      = max(0, len(path_verts_p) - 2)
    print(f"Cross-floor shortest path  ({time.time()-t0:.1f}s)")
    print(f"  Length : {path_len:.1f}")
    print(f"  Edges  : {len(path_edges_p)}")
    print(f"  Turns  : {n_turns}")

    kv = [path_verts_p[0]]
    for i in range(1, len(path_verts_p) - 1):
        z_prev = Vertex.Z(path_verts_p[i - 1])
        z_curr = Vertex.Z(path_verts_p[i])
        z_next = Vertex.Z(path_verts_p[i + 1])
        if abs(z_curr - z_prev) > 0.1 or abs(z_curr - z_next) > 0.1:
            kv.append(path_verts_p[i])
    kv.append(path_verts_p[-1])
    seen, key_verts = set(), []
    for v in kv:
        k = (round(Vertex.X(v), 1), round(Vertex.Y(v), 1), round(Vertex.Z(v), 1))
        if k not in seen:
            seen.add(k); key_verts.append(v)

    simp_len = sum(
        math.sqrt((Vertex.X(key_verts[i+1]) - Vertex.X(key_verts[i]))**2 +
                  (Vertex.Y(key_verts[i+1]) - Vertex.Y(key_verts[i]))**2 +
                  (Vertex.Z(key_verts[i+1]) - Vertex.Z(key_verts[i]))**2)
        for i in range(len(key_verts) - 1))
    reduction = (path_len - simp_len) / path_len * 100 if path_len else 0.0
    print(f"\nSimplified path (stair waypoints only):")
    print(f"  Length    : {simp_len:.1f}  (-{reduction:.1f}% vs full path)")
    print(f"  Waypoints : {len(key_verts)}  (from {len(path_verts_p)} nodes)")

# Figure
fig = go.Figure()
for fi, lv in enumerate(FLOOR_LEVELS):
    tris    = floor_tris[lv]
    z_level = fi * FLOOR_HEIGHT
    vx, vy, vz, ii, jj, kk = [], [], [], [], [], []
    for ti, t in enumerate(tris):
        base = ti * 3
        vx += [t[0, 0], t[1, 0], t[2, 0]]
        vy += [t[0, 1], t[1, 1], t[2, 1]]
        vz += [z_level, z_level, z_level]
        ii.append(base); jj.append(base + 1); kk.append(base + 2)
    fig.add_trace(go.Mesh3d(x=vx, y=vy, z=vz, i=ii, j=jj, k=kk,
                            opacity=0.13, color="steelblue", showscale=False,
                            showlegend=False, hoverinfo="skip", flatshading=True))

fig.add_trace(go.Scatter3d(x=[Vertex.X(v) for v in gverts], y=[Vertex.Y(v) for v in gverts],
                           z=[Vertex.Z(v) for v in gverts], mode="markers",
                           marker=dict(size=1.5, color="rgba(160,180,220,0.3)"),
                           showlegend=False, hoverinfo="skip"))

for (a, b) in stair_node_pairs:
    pa, pb = all_v[a], all_v[b]
    fig.add_trace(go.Scatter3d(x=[Vertex.X(pa), Vertex.X(pb)], y=[Vertex.Y(pa), Vertex.Y(pb)],
                               z=[Vertex.Z(pa), Vertex.Z(pb)], mode="lines",
                               line=dict(color="rgba(255,100,50,0.4)", width=2),
                               showlegend=False, hoverinfo="skip"))

if path is not None:
    pv = Topology.Vertices(path)
    fig.add_trace(go.Scatter3d(x=[Vertex.X(v) for v in pv], y=[Vertex.Y(v) for v in pv],
                               z=[Vertex.Z(v) for v in pv], mode="lines+markers",
                               line=dict(color="red", width=7), marker=dict(size=3, color="red"),
                               name=f"Shortest path  L={path_len:.0f}"))
if key_verts is not None:
    fig.add_trace(go.Scatter3d(x=[Vertex.X(v) for v in key_verts], y=[Vertex.Y(v) for v in key_verts],
                               z=[Vertex.Z(v) for v in key_verts], mode="lines+markers",
                               line=dict(color="#00ff88", width=6, dash="dot"),
                               marker=dict(size=7, color="#00ff88", symbol="diamond"),
                               name=f"Simplified  L={simp_len:.0f}  ({len(key_verts)} pts)"))
if path is not None:
    for v, c, lbl in [(start_v, "cyan", "Start"), (end_v, "orange", "End")]:
        fig.add_trace(go.Scatter3d(x=[Vertex.X(v)], y=[Vertex.Y(v)], z=[Vertex.Z(v)],
                                   mode="markers+text", marker=dict(size=13, color=c, symbol="diamond"),
                                   text=[lbl], textposition="top center",
                                   textfont=dict(color=c, size=13), showlegend=False))

fig.update_layout(paper_bgcolor="black",
    scene=dict(aspectmode="data",
               xaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="x", font=dict(color="white"))),
               yaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="y", font=dict(color="white"))),
               zaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="z (floor stack)", font=dict(color="white")))),
    scene_camera=dict(projection=dict(type="orthographic"), eye=dict(x=1.4, y=1.4, z=1.4)),
    scene_dragmode="orbit",
    legend=dict(font=dict(color="white", size=13), bgcolor="rgba(0,0,0,0.65)", x=0.01, y=0.99),
    width=1400, height=900)
fig.show(renderer=INTERACTIVE_RENDERER)
save_fig(fig, "08_shortest_path_3d.png")

## 16. Building-wide DEGREE CENTRALITY

Degree centrality on the **whole-building graph** (both floors connected through the stairs),
so stair nodes and well-connected circulation spaces score across floors — not per plan in
isolation.

In [ ]:
t0 = time.time()
degree_values = Graph.DegreeCentrality(building_graph, normalize=True)
n_nodes = len(degree_values)
a = np.array(degree_values, dtype=float)
print(f"Degree centrality — {n_nodes} nodes  ({time.time()-t0:.1f}s)")
print(f"  Range : {a.min():.4f} - {a.max():.4f}")
print(f"  Mean  : {a.mean():.4f}   Std: {a.std():.4f}")

sorted_dc = sorted(zip(gverts, degree_values), key=lambda x: x[1], reverse=True)
print("\nTop 5 most connected (hubs):")
for v, score in sorted_dc[:5]:
    fl = int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1
    links = round(score * (n_nodes - 1))
    print(f"  Floor {fl}  ({Vertex.X(v):.1f}, {Vertex.Y(v):.1f})  ->  {score:.4f}  ({links} connections)")
print("\nTop 5 least connected (dead ends):")
for v, score in sorted_dc[-5:]:
    fl = int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1
    links = round(score * (n_nodes - 1))
    print(f"  Floor {fl}  ({Vertex.X(v):.1f}, {Vertex.Y(v):.1f})  ->  {score:.4f}  ({links} connections)")

heatmap_from_graph(degree_values, "Degree Centrality (whole building)",
                   "04_degree_centrality.png", "viridis")

## 17. Closeness Centrality (Integration)

How close each space is to every other space in the **entire building**. High values =
globally integrated, easy-to-reach locations.

In [ ]:
t0 = time.time()
closeness_values = Graph.ClosenessCentrality(building_graph)
a = np.array(closeness_values, dtype=float)
print(f"Closeness centrality — {len(closeness_values)} nodes  ({time.time()-t0:.1f}s)")
print(f"  Range : {a.min():.4f} - {a.max():.4f}")
print(f"  Mean  : {a.mean():.4f}   Std: {a.std():.4f}")

sorted_cc = sorted(zip(gverts, closeness_values), key=lambda x: x[1], reverse=True)
print("\nTop 5 most integrated (closest to all):")
for v, score in sorted_cc[:5]:
    fl = int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1
    print(f"  Floor {fl}  ({Vertex.X(v):.1f}, {Vertex.Y(v):.1f})  ->  {score:.4f}")

heatmap_from_graph(closeness_values, "Closeness Centrality / Integration (whole building)",
                   "05_closeness_centrality.png", "thermal")

## 18. Betweenness Centrality (Choice)

How often each space lies on the shortest paths between all other spaces. The stair nodes
light up because every cross-floor trip passes through them.

In [ ]:
t0 = time.time()
betweenness_values = Graph.BetweennessCentrality(building_graph, normalize=True)
a = np.array(betweenness_values, dtype=float)
print(f"Betweenness centrality — {len(betweenness_values)} nodes  ({time.time()-t0:.1f}s)")
print(f"  Range : {a.min():.4f} - {a.max():.4f}")
print(f"  Mean  : {a.mean():.4f}   Std: {a.std():.4f}")

sorted_bc = sorted(zip(gverts, betweenness_values), key=lambda x: x[1], reverse=True)
print("\nTop 5 most traversed spaces (critical paths):")
for v, score in sorted_bc[:5]:
    fl = int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1
    print(f"  Floor {fl}  ({Vertex.X(v):.1f}, {Vertex.Y(v):.1f})  ->  {score:.4f}")

heatmap_from_graph(betweenness_values, "Betweenness Centrality / Choice (whole building)",
                   "06_betweenness_centrality.png", "thermal")

## 19. Community Detection

Partition the building into spatial communities — densely connected groups of nodes. Because
the graph spans both floors, a community can extend vertically through a stair.

In [ ]:
from collections import Counter
t0 = time.time()
community_values = Graph.CommunityPartition(building_graph)
n_comm = len(set(community_values))
community_counts = Counter(community_values)
cell_area_m2 = GRID_SIZE ** 2

print(f"Detected {n_comm} communities  ({time.time()-t0:.1f}s)")
print(f"\n{'ID':>4}  {'Cells':>6}  {'Area m2':>9}")
for cid, cnt in sorted(community_counts.items()):
    print(f"  {cid:>3}    {cnt:>5}    {cnt * cell_area_m2:>7.2f}")

largest_id  = max(community_counts, key=community_counts.get)
smallest_id = min(community_counts, key=community_counts.get)
total_area  = len(community_values) * cell_area_m2
print(f"\nLargest:  Community {largest_id} — {community_counts[largest_id]} cells, "
      f"~{community_counts[largest_id] * cell_area_m2:.2f} m2")
print(f"Smallest: Community {smallest_id} — {community_counts[smallest_id]} cells, "
      f"~{community_counts[smallest_id] * cell_area_m2:.2f} m2")
print(f"Total navigable cells: {len(community_values)},  area ~{total_area:.2f} m2")

heatmap_from_graph(community_values, f"Community Detection — {n_comm} communities (whole building)",
                   "07_communities.png", "rainbow")

## 20. Visibility Heatmap / VGA (per floor)

Visibility does **not** cross floors, so this Visibility Graph Analysis runs **independently
per floor**. A coarse grid of viewpoints is connected wherever the sightline stays inside the
floor; the **visibility degree** = how many other viewpoints each one can see.

In [ ]:
VGA_GRID_SIZE = max(GRID_SIZE, 2.0)   # coarser than the analysis grid (O(n^2) cost)
VIS_SAMPLES   = 12                    # samples along each sightline for the occlusion test

def compute_visibility(tris, viewpoints):
    n = len(viewpoints)
    ts = np.linspace(0.12, 0.88, VIS_SAMPLES)
    deg = np.zeros(n, int)
    for i in range(n):
        for j in range(i + 1, n):
            seg = viewpoints[i][None, :] * (1 - ts)[:, None] + viewpoints[j][None, :] * ts[:, None]
            if points_inside(tris, seg).all():
                deg[i] += 1; deg[j] += 1
    return deg

uvp = np.arange(UMIN, UMAX + VGA_GRID_SIZE, VGA_GRID_SIZE)
vvp = np.arange(VMIN, VMAX + VGA_GRID_SIZE, VGA_GRID_SIZE)
UUv, VVv = np.meshgrid(uvp, vvp)
VP_PTS = np.column_stack([UUv.ravel(), VVv.ravel()])

per_floor_vga = []
for fi, lv in enumerate(FLOOR_LEVELS):
    tris = floor_tris[lv]
    vp = VP_PTS[points_inside(tris, VP_PTS)]
    t0 = time.time()
    deg = compute_visibility(tris, vp)
    per_floor_vga.append((vp[:, 0], vp[:, 1], deg))
    print(f"  {FLOOR_NAMES[fi]}: {len(vp)} viewpoints, "
          f"visibility {deg.min() if len(deg) else 0}-{deg.max() if len(deg) else 0} "
          f"(mean {deg.mean():.1f}, {time.time()-t0:.1f}s)")

show_grid_heatmaps(per_floor_vga, "Visibility Graph Analysis - isovist degree (per floor)",
                   "09_visibility_isovist.png", "plasma", grid_size=VGA_GRID_SIZE)

## 21. Isovist Analysis (both floors)

An **isovist** is the polygon of space visible from a single viewpoint. Using ray-casting over
the navigable cell mask (robust on every floor): cast rays from each viewpoint and march each
ray outward until it leaves the navigable mask. The ring of stop-points is the isovist polygon.

In [ ]:
ISO_STEP   = max(GRID_SIZE, 2.0)   # viewpoint spacing (coarse, so overlaid isovists stay readable)
ISO_NRAYS  = 50                    # rays cast per viewpoint
ISO_RSTEP  = GRID_SIZE             # marching step along each ray

dw = UMAX - UMIN; dh = VMAX - VMIN
MAX_R = float(np.hypot(dw, dh))
ux_m = np.arange(UMIN, UMAX + GRID_SIZE, GRID_SIZE)
uy_m = np.arange(VMIN, VMAX + GRID_SIZE, GRID_SIZE)

def build_mask(valid_xy):
    m = np.zeros((len(uy_m), len(ux_m)), bool)
    xi = np.round((valid_xy[:, 0] - UMIN) / GRID_SIZE).astype(int)
    yi = np.round((valid_xy[:, 1] - VMIN) / GRID_SIZE).astype(int)
    ok = (xi >= 0) & (xi < m.shape[1]) & (yi >= 0) & (yi < m.shape[0])
    m[yi[ok], xi[ok]] = True
    return m

def inside_mask(pts, m):
    xi = np.round((pts[:, 0] - UMIN) / GRID_SIZE).astype(int)
    yi = np.round((pts[:, 1] - VMIN) / GRID_SIZE).astype(int)
    ok = (xi >= 0) & (xi < m.shape[1]) & (yi >= 0) & (yi < m.shape[0])
    res = np.zeros(len(pts), bool)
    res[ok] = m[yi[ok], xi[ok]]
    return res

_angles = np.linspace(0, 2 * np.pi, ISO_NRAYS, endpoint=False)
_dirs   = np.stack([np.cos(_angles), np.sin(_angles)], axis=1)
_rs     = np.arange(1, int(MAX_R / ISO_RSTEP) + 1) * ISO_RSTEP

def isovist_polygon(origin, m):
    P = origin[None, None, :] + _dirs[:, None, :] * _rs[None, :, None]
    ins = inside_mask(P.reshape(-1, 2), m).reshape(len(_dirs), len(_rs))
    cont   = np.cumprod(ins, axis=1).astype(bool)
    counts = cont.sum(1)
    radius = np.where(counts > 0, _rs[np.clip(counts - 1, 0, len(_rs) - 1)], ISO_RSTEP * 0.3)
    return origin[None, :] + _dirs * radius[:, None]

uvp = np.arange(UMIN, UMAX + ISO_STEP, ISO_STEP)
vvp = np.arange(VMIN, VMAX + ISO_STEP, ISO_STEP)
UUv, VVv = np.meshgrid(uvp, vvp)
VP_ALL = np.column_stack([UUv.ravel(), VVv.ravel()])

fig = make_subplots(rows=1, cols=len(FLOOR_LEVELS), horizontal_spacing=0.06,
                    subplot_titles=[FLOOR_NAMES[i] for i in range(len(FLOOR_LEVELS))])
iso_means = []
for fi, lv in enumerate(FLOOR_LEVELS):
    m = build_mask(floor_valid[lv])
    vp = VP_ALL[inside_mask(VP_ALL, m)]
    t0 = time.time()
    Zmask = np.where(m, 1.0, np.nan)
    fig.add_trace(go.Heatmap(x=ux_m, y=uy_m, z=Zmask, showscale=False, hoverongaps=False,
                             colorscale=[[0, "rgba(150,150,150,0.30)"], [1, "rgba(150,150,150,0.30)"]]),
                  row=1, col=fi + 1)
    areas = []
    for o in vp:
        poly = isovist_polygon(np.asarray(o, float), m)
        xs = list(poly[:, 0]) + [poly[0, 0]]
        ys = list(poly[:, 1]) + [poly[0, 1]]
        ar = 0.5 * abs(np.dot(poly[:, 0], np.roll(poly[:, 1], -1)) -
                       np.dot(poly[:, 1], np.roll(poly[:, 0], -1)))
        areas.append(ar)
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", fill="toself",
                                 line=dict(color="rgba(0,220,255,0.45)", width=1),
                                 fillcolor="rgba(0,220,255,0.10)", showlegend=False),
                      row=1, col=fi + 1)
    if len(vp):
        fig.add_trace(go.Scatter(x=vp[:, 0], y=vp[:, 1], mode="markers",
                                 marker=dict(size=4, color="yellow"), showlegend=False),
                      row=1, col=fi + 1)
    fig.update_yaxes(scaleanchor=f"x{fi+1 if fi else ''}", scaleratio=1, row=1, col=fi + 1)
    amean = (sum(areas) / len(areas)) if areas else 0.0
    iso_means.append(amean)
    print(f"  {FLOOR_NAMES[fi]}: {len(vp)} viewpoints, mean isovist area {amean:.1f} m2 ({time.time()-t0:.1f}s)")

px_pu = max(8, min(40, int(1500 / (dw * len(FLOOR_LEVELS) + 1))))
fig_w = max(700, int(dw * px_pu) * len(FLOOR_LEVELS) + 300)
fig_h = max(420, int(dh * px_pu) + 170)
fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
for ann in fig.layout.annotations:
    ann.font.color = "white"
fig.update_layout(title=dict(text="Isovist Analysis (both floors)", font=dict(color="white")),
                  height=fig_h, width=fig_w, paper_bgcolor="black", plot_bgcolor="black")
fig.show(renderer=renderer)
save_fig(fig, "11_isovists_all_floors.png")

## 22. Building-wide Summary

All values come from the single connected building graph, so they describe the *whole* stacked
Dubai Tower section rather than any individual floor.

In [ ]:
def per_floor_mean(values):
    fl = np.array([int(round(Vertex.Z(v) / FLOOR_HEIGHT)) for v in gverts])
    vals = np.array(values, dtype=float)
    return [round(float(vals[fl == i].mean()), 4) if np.any(fl == i) else None
            for i in range(len(FLOOR_LEVELS))]

print(f"{'Metric':<24}{'min':>10}{'max':>10}{'mean':>10}   per-floor mean")
for name, vals in [("Degree centrality", degree_values),
                   ("Closeness centrality", closeness_values),
                   ("Betweenness centrality", betweenness_values)]:
    a = np.array(vals, dtype=float)
    print(f"{name:<24}{a.min():>10.4f}{a.max():>10.4f}{a.mean():>10.4f}   {per_floor_mean(vals)}")
print()
print(f"Nodes: {len(gverts)} | Edges: {len(gedges)} | Density: {Graph.Density(building_graph):.5f} "
      f"| Communities: {len(set(community_values))} | Stair edges: {len(stair_node_pairs)}")

## 23. Export analysis summary (notes + metadata)

Collect every metric into a human-readable **`analysis_summary.md`** and a machine-readable
**`analysis_metadata.json`**, both written to `ASSETS_DIR`. Uses `globals()` guards so it is
safe to run after a partial execution.

In [ ]:
import json as _json
from datetime import datetime

def _stats(arr):
    a = np.asarray(arr, dtype=float)
    if a.size == 0:
        return dict(n=0, min=None, max=None, mean=None, std=None)
    return dict(n=int(a.size), min=float(a.min()), max=float(a.max()),
                mean=float(a.mean()), std=float(a.std()))

def _per_floor_mean(values):
    fl = np.array([int(round(Vertex.Z(v) / FLOOR_HEIGHT)) for v in gverts])
    vals = np.array(values, dtype=float)
    return [round(float(vals[fl == i].mean()), 6) if np.any(fl == i) else None
            for i in range(len(FLOOR_LEVELS))]

G = globals()
meta = {}
meta["generated"]   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
meta["notebook"]    = "NB_DubaiTower_MultiFloor_Spatial_Intelligence.ipynb"
meta["description"] = "Spatial Intelligence analysis of two stacked residential floors (Level 03 + Level 04) of the Dubai Tower as one connected building."

meta["parameters"] = {
    "FLOOR_TAGS": FLOOR_TAGS,
    "FLOOR_OBJS": FLOOR_OBJS,
    "GRID_SIZE": GRID_SIZE,
    "FLOOR_HEIGHT": FLOOR_HEIGHT,
    "FLOOR_LEVELS": FLOOR_LEVELS,
    "PLAN_AXES": list(PLAN_AXES),
    "VGA_GRID_SIZE": G.get("VGA_GRID_SIZE"),
    "VIS_SAMPLES": G.get("VIS_SAMPLES"),
    "ISO_STEP": G.get("ISO_STEP"),
    "n_stair_locations": len(stairs_used),
    "stairs_auto_placed": not bool(STAIR_LOCATIONS),
}

meta["geometry"] = {
    "plan_bbox": {"x_min": float(UMIN), "x_max": float(UMAX),
                  "y_min": float(VMIN), "y_max": float(VMAX),
                  "width": float(UMAX - UMIN), "height": float(VMAX - VMIN)},
    "faces_per_floor": {FLOOR_NAMES[i]: len(floor_faces[lv]) for i, lv in enumerate(FLOOR_LEVELS)},
    "navigable_nodes_per_floor": {FLOOR_NAMES[i]: int(len(floor_valid[lv])) for i, lv in enumerate(FLOOR_LEVELS)},
}

meta["graph"] = {
    "nodes": len(gverts), "edges": len(gedges),
    "density": float(Graph.Density(building_graph)), "stair_edges": len(stair_node_pairs),
}

if "mst" in G:
    meta["mst"] = {"vertices": len(mst_verts), "edges": len(mst_edges), "density": float(Graph.Density(mst))}

if G.get("path") is not None:
    meta["shortest_path"] = {
        "length": float(path_len), "n_nodes": len(Topology.Vertices(path)),
        "simplified_length": float(simp_len) if "simp_len" in G else None,
        "simplified_waypoints": len(key_verts) if G.get("key_verts") else None,
    }

meta["centrality"] = {}
for name, key in [("degree_values", "degree"), ("closeness_values", "closeness"),
                  ("betweenness_values", "betweenness")]:
    if name in G:
        s = _stats(G[name]); s["per_floor_mean"] = _per_floor_mean(G[name])
        meta["centrality"][key] = s

if "degree_values" in G:
    n_nodes = len(degree_values)
    top = sorted(zip(gverts, degree_values), key=lambda x: x[1], reverse=True)[:5]
    meta["centrality"]["top5_degree_hubs"] = [
        {"floor": int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1,
         "x": round(Vertex.X(v), 2), "y": round(Vertex.Y(v), 2),
         "score": round(float(sc), 6), "connections": round(float(sc) * (n_nodes - 1))}
        for v, sc in top]

if "community_values" in G:
    from collections import Counter as _Counter
    cc = _Counter(community_values); cell_area = GRID_SIZE ** 2
    meta["communities"] = {
        "count": len(set(community_values)), "total_cells": len(community_values),
        "total_area_m2": round(len(community_values) * cell_area, 3),
        "sizes": {str(cid): {"cells": int(cnt), "area_m2": round(cnt * cell_area, 3)}
                  for cid, cnt in sorted(cc.items())},
    }

if "per_floor_vga" in G:
    meta["visibility_vga"] = {}
    for i, (xs, ys, deg) in enumerate(per_floor_vga):
        d = np.asarray(deg, dtype=float)
        meta["visibility_vga"][FLOOR_NAMES[i]] = {
            "viewpoints": int(d.size), "min": int(d.min()) if d.size else None,
            "max": int(d.max()) if d.size else None,
            "mean": round(float(d.mean()), 3) if d.size else None}

json_path = os.path.join(ASSETS_DIR, "analysis_metadata.json")
with open(json_path, "w", encoding="utf-8") as f:
    _json.dump(meta, f, ensure_ascii=False, indent=2)
print(f"Saved: {json_path}")

L = []
L.append(f"# Analysis Summary — Dubai Tower (Level 03 + Level 04)\n")
L.append(f"_Generated: {meta['generated']}_\n")
L.append(meta["description"] + "\n")
L.append("## Parameters\n")
L.append("| Parameter | Value |"); L.append("|---|---|")
for k, v in meta["parameters"].items():
    L.append(f"| `{k}` | {v} |")
L.append("")
g = meta["geometry"]["plan_bbox"]
L.append("## Geometry\n")
L.append(f"- **Plan bounding box:** X [{g['x_min']:.2f}, {g['x_max']:.2f}], "
         f"Y [{g['y_min']:.2f}, {g['y_max']:.2f}]  ({g['width']:.2f} x {g['height']:.2f} units)\n")
L.append("| Floor | Faces | Navigable nodes |"); L.append("|---|---|---|")
for i, lv in enumerate(FLOOR_LEVELS):
    fn = FLOOR_NAMES[i]
    L.append(f"| {fn} | {meta['geometry']['faces_per_floor'][fn]} | "
             f"{meta['geometry']['navigable_nodes_per_floor'][fn]} |")
L.append("")
gr = meta["graph"]
L.append("## Building graph\n")
L.append(f"- Nodes: **{gr['nodes']}**  |  Edges: **{gr['edges']}**  |  "
         f"Density: **{gr['density']:.5f}**  |  Stair edges: **{gr['stair_edges']}**\n")
if "mst" in meta:
    m = meta["mst"]; L.append("## Minimum Spanning Tree\n")
    L.append(f"- Vertices: {m['vertices']}  |  Edges: {m['edges']}  |  Density: {m['density']:.5f}\n")
if "shortest_path" in meta:
    sp = meta["shortest_path"]; L.append("## Cross-floor shortest path\n")
    L.append(f"- Length: **{sp['length']:.1f}**  |  Nodes: {sp['n_nodes']}")
    if sp.get("simplified_length") is not None:
        L.append(f"  |  Simplified: {sp['simplified_length']:.1f} ({sp['simplified_waypoints']} waypoints)")
    L.append("")
if meta.get("centrality"):
    L.append("## Centrality metrics\n")
    L.append("| Metric | min | max | mean | std | per-floor mean |"); L.append("|---|---|---|---|---|---|")
    for key in ("degree", "closeness", "betweenness"):
        s = meta["centrality"].get(key)
        if s:
            L.append(f"| {key.capitalize()} | {s['min']:.4f} | {s['max']:.4f} | "
                     f"{s['mean']:.4f} | {s['std']:.4f} | {s['per_floor_mean']} |")
    L.append("")
    if "top5_degree_hubs" in meta["centrality"]:
        L.append("**Top 5 degree hubs:**\n")
        for h in meta["centrality"]["top5_degree_hubs"]:
            L.append(f"- Floor {h['floor']} ({h['x']}, {h['y']}) -> {h['score']:.4f} ({h['connections']} connections)")
        L.append("")
if "communities" in meta:
    cm = meta["communities"]; L.append("## Community detection\n")
    L.append(f"- Communities: **{cm['count']}**  |  Total cells: {cm['total_cells']}  |  "
             f"Total area: ~{cm['total_area_m2']} m2\n")
    L.append("| Community | Cells | Area m2 |"); L.append("|---|---|---|")
    for cid, d in cm["sizes"].items():
        L.append(f"| {cid} | {d['cells']} | {d['area_m2']} |")
    L.append("")
if "visibility_vga" in meta:
    L.append("## Visibility Graph Analysis (per floor)\n")
    L.append("| Floor | Viewpoints | min | max | mean |"); L.append("|---|---|---|---|---|")
    for fn, d in meta["visibility_vga"].items():
        L.append(f"| {fn} | {d['viewpoints']} | {d['min']} | {d['max']} | {d['mean']} |")
    L.append("")
L.append("---")
L.append("_Generated automatically by section 23. Community colours/partition are stochastic and may differ between runs._")

md_path = os.path.join(ASSETS_DIR, "analysis_summary.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(L))
print(f"Saved: {md_path}")
print("\n" + "\n".join(L[:18]))